# DLVA: YOLOv8n + DINOv2 fusion, cheaper-inference experiments
Runs in priority order, one run per GPU, 2 seeds each:
1. **Distillation:** plain YOLOv8n trained to predict DINOv2's P5 features; no ViT at inference.
2. **Shorter DINOv2:** LN+GN fusion using only the first 4 or the first 8 of the 12 ViT blocks.

A run only starts if it is expected to finish before the 12 h session limit. Each finished run is copied to `/kaggle/working/out/runs/<group>/<name>`.

In [ ]:
import os, subprocess, sys, time
T0 = time.time()
DEADLINE = T0 + 11.3 * 3600  # Kaggle stops the session at 12 h; started runs must end before this
REPO, BRANCH = 'https://github.com/mshanthasuresh/YOLOv8-DINOv2.git', 'dlva-final-project'
ROOT, OUT = '/kaggle/temp/nafuse', '/kaggle/working/out'  # datasets stay in temp; only results become output
os.makedirs(f'{OUT}/logs', exist_ok=True)
if not os.path.exists(ROOT):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, ROOT], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.142', 'faster-coco-eval'], check=True)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

In [ ]:
subprocess.run([sys.executable, '-m', 'tools.prepare_data'], check=True)
# Fetch shared weights once, so parallel workers never download the same file at the same time.
subprocess.run([sys.executable, '-c', "from ultralytics import YOLO; YOLO('yolov8n.pt'); "
                "from nafuse.dino import load_backbone; load_backbone('dinov2_vits14')"], check=True)

In [ ]:
import signal
QUEUE = [
    'voc-yolov8n-distill-s0',   # YOLOv8n distilled from DINOv2: no ViT at inference
    'voc-yolov8n-lngn-d4-s0',   # fusion with the first 4 of 12 DINOv2 blocks
    'voc-yolov8n-lngn-d8-s0',   # first 8 of 12 blocks
    'voc-yolov8n-distill-s1',
    'voc-yolov8n-lngn-d4-s1',
    'voc-yolov8n-lngn-d8-s1',
]
gpus = [l for l in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines() if l.startswith('GPU')]
workers = []
for g in range(max(1, len(gpus))):
    workers.append(subprocess.Popen(
        [sys.executable, '-m', 'tools.kaggle_worker', '--deadline', str(DEADLINE), '--out', OUT, *QUEUE],
        env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}, start_new_session=True,
        stdout=open(f'{OUT}/logs/worker{g}.log', 'w'), stderr=subprocess.STDOUT))
    time.sleep(180)  # let the first worker build the dataset label caches alone
for p in workers:
    try:
        p.wait(timeout=max(1, DEADLINE + 1200 - time.time()))
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGKILL)  # an overrunning run must not cost the finished outputs

In [ ]:
import glob, json
for path in sorted(glob.glob(f'{OUT}/runs/*/*/result.json')):
    r = json.load(open(path))
    print(f"{r['spec']['name']:32s} test mAP50-95 {100 * r['test']['map']:.2f}")
for log in sorted(glob.glob(f'{OUT}/logs/worker*.log')):
    print(log)
    print(''.join(l for l in open(log, errors='ignore') if l.startswith('[')))
print(f'session time {(time.time() - T0) / 3600:.2f} h')